# Phase 10 - Reproducibility and validation

This notebook presents the consolidated, non-destructive audit of Phases 1-9. The command-line Phase 10 runner executes every workflow notebook in a separate fresh kernel before executing this notebook. It preserves all raw and derived files and validates the local source snapshots, processing manifests, data registers, CRS/datum declarations, confidence fields, figures and safety constraints.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from uqgems.phase10 import run_phase10

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

summary = run_phase10(
    PROJECT_ROOT,
    execute_notebooks=False,
    execute_validation_notebook=False,
)
print(f"Status: {summary['status']}")
print(
    f"Prior checks: {summary['prior_phase_validation']['checks_passed']}/"
    f"{summary['prior_phase_validation']['checks_total']}"
)
print(
    f"Recorded fresh-kernel notebooks: {summary['notebook_execution']['passed']}/"
    f"{summary['notebook_execution']['expected']}"
)
print(f"Phase 10 checks: {sum(summary['checks'].values())}/{len(summary['checks'])}")
print(summary["decision_gate"])

## Fresh-kernel execution evidence

Each row has a unique kernel-session identifier. When the CLI is currently executing this validation notebook, its own tenth row is appended after this kernel exits; the final CSV and JSON report then record all ten notebooks.

In [ ]:
notebooks = pd.read_csv(PROJECT_ROOT / "reports/tables/phase10_notebook_execution.csv")
display(
    notebooks[
        [
            "phase",
            "label",
            "source_notebook",
            "kernel",
            "duration_seconds",
            "code_cells",
            "executed_code_cells",
            "error_outputs",
            "status",
        ]
    ]
)

## Prior phase and requirements gates

In [ ]:
phase_results = pd.read_csv(PROJECT_ROOT / "reports/tables/phase10_phase_results.csv")
requirements = pd.read_csv(PROJECT_ROOT / "reports/tables/phase10_requirements_matrix.csv")
display(
    phase_results[
        [
            "phase",
            "status",
            "checks_passed",
            "checks_total",
            "cache_reused",
            "all_declared_outputs_exist",
        ]
    ]
)
display(requirements)

## Provenance and integrity

The deferred UQ-controlled dataset is a documented planned source, not a missing acquired file. Every acquired local source must exist and match its registered SHA-256 value. Core derived outputs must match their processing manifests and remain byte-identical across the cached replay.

In [ ]:
sources = pd.read_csv(PROJECT_ROOT / "reports/tables/phase10_source_register_audit.csv")
hashes = pd.read_csv(PROJECT_ROOT / "reports/tables/phase10_output_hash_audit.csv")
display(
    sources.loc[
        sources["deferred_source"],
        [
            "dataset",
            "processing_status",
            "status",
        ],
    ]
)
display(
    hashes.loc[
        ~(hashes["matches_manifest"] & hashes["unchanged_by_replay"]),
        ["path", "matches_manifest", "unchanged_by_replay"],
    ]
)

## Consolidated validation dashboard

In [ ]:
display(
    Image(
        filename=PROJECT_ROOT / "reports/figures/phase10_validation_dashboard.png",
        width=1100,
    )
)

## Interpretation

A passing result means the pilot can be reproduced from the preserved local source snapshots and validated cached products without hidden notebook state. It does not prove that live external downloads remain available, and it does not make the sparse, unknown-depth public utility layer survey-grade. The integrated model remains a research demonstration and evidence for requesting better UQ or asset-owner data; it is not excavation-safe.